# EPICS: capture → TRELLIS.2 → scaled reference mesh (Sol / Open OnDemand)

Use the **EPICS TRELLIS.2** kernel (`bash sol/register_kernel.sh` once) in a Jupyter session with **1 A100 or H100 GPU**.

This runs the same code as `sbatch sol/run_pipeline.sbatch`. The result is reference geometry for device design and stays
`pending_human_review`; matching the entered measurements is a software check, not a clinical or fit assessment.

In [ ]:
import json, os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - start the session with a GPU")
print("ATTN_BACKEND:", os.environ.get("ATTN_BACKEND", "flash_attn"), "| HF_HOME:", os.environ.get("HF_HOME"))

## 1. Inputs
`CAPTURE_DIR` holds `capture.json` from the web app plus both original photos. `OUT_DIR` must be new or empty.

In [ ]:
from pathlib import Path
USER = os.environ["USER"]
CAPTURE_DIR = Path(f"/scratch/{USER}/captures/example")
OUT_DIR = Path(f"/scratch/{USER}/runs/example")
GLB = None          # path to an existing GLB to skip generation
OUTER_SHELL = False # keep only the outer wall of a double-walled shell (team decision)

## 2. Generation settings
Defaults match the run logged in `docs/stub_scaling.md`. Try `remesh=False` if the scaler reports a double-walled shell.

In [ ]:
from epics_geometry.trellis_run import TrellisSettings
settings = TrellisSettings(seed=42, resolution="1024", decimation_target=500_000, texture_size=2048, remesh=True)
settings

## 3. Run

In [ ]:
from epics_geometry.pipeline import run
manifest = run(OUT_DIR, capture_dir=CAPTURE_DIR, glb=GLB, settings=settings, outer_shell=OUTER_SHELL)
print("status:", manifest["status"], "| failed stage:", manifest["failed_stage"])
for name, stage in manifest["stages"].items():
    print(f"  {name:<12}", stage.get("status") or stage.get("overall_status") or ("skipped" if stage.get("skipped") else "done"))
if manifest.get("hint"): print("hint:", manifest["hint"])
if manifest.get("error"): print(manifest["error"]["traceback"])

## 4. Reports

In [ ]:
scale_report = OUT_DIR / "scale_report.json"
if scale_report.exists():
    r = json.loads(scale_report.read_text())
    print(r["status"], "-", r["message"])
    for x in r.get("results", []):
        print(f"  {'ok ' if x['within_tolerance'] else 'OUT'} {x['measurement']:<18} target {x['target_mm']:8.2f}  actual {x['actual_mm']:8.2f}")
validation = OUT_DIR / "validation.json"
if validation.exists():
    print(json.loads(validation.read_text())["summary"])

## 5. Preview
Download `scaled.glb` (metres, base at Z=0, tip toward +Z) and `manifest.json` for review and Rhino import.

In [ ]:
import trimesh
result = manifest.get("result_glb") or ({"path": str(OUT_DIR / "generated.glb")} if (OUT_DIR / "generated.glb").exists() else None)
trimesh.load(result["path"]).show() if result else print("no mesh to show")